# Exercise 2 (Instructor Solution) — Domain Feature Engineering & Leak-Free `ColumnTransformer`

Complete reference solution for Exercise 2.


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cust = pd.read_csv("data/customers_raw.csv")
with open("data/usage_events.json", "r", encoding="utf-8") as f:
    usage = pd.DataFrame(json.load(f))
cust["plan_tier"] = cust["plan_tier"].apply(lambda x: x.strip().lower() if isinstance(x, str) else x)
cust["country"] = cust["country"].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)
df = cust.drop_duplicates(subset=["customer_id"], keep="first").merge(usage, on="customer_id", how="left")


In [ ]:
def engineer_churn_features(input_df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy of input_df with 3 new domain features."""
    out = input_df.copy()
    out["charges_per_login"] = out["monthly_charges"] / (out["login_days_30d"] + 1.0)
    out["high_ticket_flag"] = (out["support_tickets_30d"] >= 3).astype(int)
    out["engagement_ratio"] = out["watch_hours_30d"] / (out["monthly_charges"] + 1.0)
    return out


NUMERIC_COLS = [
    "age", "tenure_months", "monthly_charges",
    "login_days_30d", "watch_hours_30d", "support_tickets_30d",
    "auto_pay", "charges_per_login", "high_ticket_flag", "engagement_ratio",
]
CATEGORICAL_COLS = ["plan_tier", "country", "payment_method", "last_active_device"]


def build_preprocessor(numeric_cols: list[str], categorical_cols: list[str]) -> ColumnTransformer:
    """Create a ColumnTransformer for numeric + categorical features."""
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ]
    )


In [ ]:
df_fe = engineer_churn_features(df)
X = df_fe[NUMERIC_COLS + CATEGORICAL_COLS]
y = df_fe["churned"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
preprocessor = build_preprocessor(NUMERIC_COLS, CATEGORICAL_COLS)
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

assert {"charges_per_login", "high_ticket_flag", "engagement_ratio"}.issubset(df_fe.columns)
assert set(df_fe["high_ticket_flag"].unique()).issubset({0, 1})
assert X_train_t.shape[0] == len(X_train)
assert X_test_t.shape[0] == len(X_test)
assert X_train_t.shape[1] == X_test_t.shape[1]
assert not np.isnan(X_train_t).any()
assert not np.isnan(X_test_t).any()

probe = X_test.iloc[[0]].copy()
probe["plan_tier"] = "future_tier_2027"
probe["monthly_charges"] = np.nan
probe_t = preprocessor.transform(probe)
assert not np.isnan(probe_t).any()

print(f"[ALL TESTS PASSED] Transformed matrix shape: train={X_train_t.shape}, test={X_test_t.shape}")
print("Output feature count:", len(preprocessor.get_feature_names_out()))
